# Option A: LLM-Based Relevance Judgements
## Moving beyond pseudo-relevance labels

**The problem with pseudo-relevance evaluation:**

Current evaluation treats a company as relevant only if the **production system**
ranked it in its top-100. This creates a ceiling — your retrieval system is penalised
for finding genuinely relevant companies that the production system missed.

**What this notebook does:**

Uses Llama-3.1-8B (local via Ollama) as a relevance judge to create proper
ground-truth labels for a held-out set of 20 queries. The LLM reads each
(query, company) pair and judges relevance independently of the production system.

**Why LLM-as-judge works:**
- LLM reads the actual query intent and company description together
- Not bounded by what the production system happened to retrieve
- Widely used in IR research when human annotation is too expensive
- Your thesis proposal explicitly described this approach

**Evaluation focus: NDCG@100**
Since Istari's goal is to deliver the best 1000 companies (not just top-10),
NDCG@100 is the primary metric — it measures whether the right companies
appear in the top 100 of your retrieved 1000.

**Held-out evaluation set:**
20 queries never seen during any training — proper generalisation test.

**Folder structure:**
```
result/
└── 08_llm_relevance_judge/
    ├── held_out_queries.json          # 20 evaluation queries
    ├── llm_judgements.json            # Raw LLM relevance scores
    ├── relevance_labels.csv           # Processed binary labels
    ├── evaluation_llm_labels.csv      # NDCG/Prec/Recall/F1 with LLM labels
    └── comparison_pseudo_vs_llm.csv   # Side-by-side comparison
```

## 1 · Environment Setup

In [ ]:
import os, json, time, random, requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RESULT_DIR      = Path('result/08_llm_relevance_judge')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
JUDGE_MODEL     = 'llama3.1:8b'

# Primary metric — NDCG@100 is most meaningful for Istari's use case
# Goal: deliver best 1000 companies, fine ranking within done separately
K_VALUES        = [10, 50, 100, 500, 1000]
PRIMARY_K       = 100

print(f'[Setup] Result folder  : {RESULT_DIR}/')
print(f'[Setup] Judge model    : {JUDGE_MODEL}')
print(f'[Setup] Primary metric : NDCG@{PRIMARY_K}')

# Ollama check
try:
    resp = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'[Setup] Ollama running : ✅  Models: {models}')
    if JUDGE_MODEL not in models:
        print(f'[Setup] WARNING: {JUDGE_MODEL} not found — run: ollama pull {JUDGE_MODEL}')
except Exception as e:
    print(f'[Setup] Ollama ERROR   : {e}')

## 2 · Imports

In [ ]:
import re
from pathlib import Path
print('[Imports] All packages loaded')

## 3 · Load Data & Define Held-Out Evaluation Set

We select 20 queries for LLM evaluation. These are held out from any training
and used only for evaluation.

**Selection strategy:** Sample diverse query types — geographic, industry-specific,
size-specific, and functional queries — to get a representative evaluation set.

The remaining 81 queries will be used for Option B (fine-tuning).

In [ ]:
print('[Load] Loading all data...')
production_df = pd.read_excel('dataset/production_results.xlsx')
all_companies = production_df.drop_duplicates(subset='domain').reset_index(drop=True)

with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Total queries    : {len(data)}')
print(f'[Load] Total companies  : {len(all_companies):,}')

# Load MiniLM results — our best baseline
minilm_df = pd.read_csv('result/03_baseline_minilm/minilm_results.csv')
print(f'[Load] MiniLM results   : {len(minilm_df):,} rows')

# ── Define held-out evaluation set — 20 queries ──────────────────────────────
# Fix random seed for reproducibility
random.seed(42)
all_qids = [item['query_id'] for item in data]

# Sample 20 held-out queries — remainder used for fine-tuning
held_out_qids = sorted(random.sample(all_qids, 20))
train_qids    = [qid for qid in all_qids if qid not in held_out_qids]

held_out_data = [item for item in data if item['query_id'] in held_out_qids]
train_data    = [item for item in data if item['query_id'] in train_qids]

print(f'\n[Load] Held-out queries : {len(held_out_data)} (for LLM evaluation)')
print(f'[Load] Training queries : {len(train_data)} (for Option B fine-tuning)')
print(f'\n[Load] Held-out query list:')
for item in held_out_data:
    print(f'  [{item["query_id"]}] {item["query"]}')

# Save held-out queries for Option B
with open(RESULT_DIR / 'held_out_queries.json', 'w') as f:
    json.dump(held_out_data, f, indent=2)
with open(RESULT_DIR / 'train_queries.json', 'w') as f:
    json.dump(train_data, f, indent=2)
print(f'\n[Load] Saved held-out and train splits')

## 4 · LLM Relevance Judge

The LLM reads each (query, company) pair and assigns a relevance score.

**Judgement scale:**
- `2` — Highly relevant: company clearly matches the query intent
- `1` — Partially relevant: company somewhat matches but not ideal
- `0` — Not relevant: company does not match the query

**Why three levels not just yes/no?**
NDCG supports graded relevance — a highly relevant company at rank 1
should score higher than a partially relevant company. This gives a more
nuanced and accurate evaluation.

**Companies judged per query:**
Top-20 from MiniLM retrieval. Judging all 1000 would take too long —
top-20 covers the positions that matter most for NDCG@10 and NDCG@100.

**Total LLM calls:** 20 queries × 20 companies = 400 calls (~10-15 minutes)

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are a relevance assessor for a company search engine.
Given a search query and a company description, assess how relevant the company
is to the search query.

Respond with ONLY a JSON object in this exact format:
{"score": <0, 1, or 2>, "reason": "<one sentence explanation>"}

Scoring:
2 = Highly relevant: company clearly matches the query intent
1 = Partially relevant: company somewhat matches but is not ideal
0 = Not relevant: company does not match the query

Be strict — only score 2 if the company is a strong, direct match."""

def judge_relevance(query, company_name, summary, max_retries=3):
    """Ask LLM to judge relevance of a company for a query."""
    # Truncate summary to keep prompt short
    summary_short = str(summary)[:400] if summary else 'No description available'

    user_msg = f"""Query: "{query}"
Company: {company_name}
Description: {summary_short}

How relevant is this company to the search query?"""

    for attempt in range(max_retries):
        try:
            response = requests.post(
                f'{OLLAMA_BASE_URL}/api/chat',
                json={
                    'model':   JUDGE_MODEL,
                    'messages': [
                        {'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_msg}
                    ],
                    'stream':  False,
                    'options': {'temperature': 0.0, 'num_predict': 80}
                },
                timeout=30
            )
            response.raise_for_status()
            raw = response.json()['message']['content'].strip()

            # Extract JSON from response
            json_match = re.search(r'\{.*\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group(0))
                score  = int(parsed.get('score', 0))
                reason = parsed.get('reason', '')
                if score in [0, 1, 2]:
                    return score, reason
            return 0, 'parse_error'

        except Exception as e:
            if attempt == max_retries - 1:
                return 0, f'error: {str(e)[:50]}'
            time.sleep(1)

# ── Quick test ────────────────────────────────────────────────────────────────
print('[Test] Testing LLM judge...')
test_q = 'software companies in Germany'
test_c = 'Software Genesis, Inc.'
test_s = 'Software Genesis is a software development company focused on the consumer market.'
score, reason = judge_relevance(test_q, test_c, test_s)
print(f'[Test] Query   : "{test_q}"')
print(f'[Test] Company : {test_c}')
print(f'[Test] Score   : {score}/2')
print(f'[Test] Reason  : {reason}')
print('[Test] LLM judge working ✅')

## 5 · Run LLM Judgements for All 20 Held-Out Queries

For each held-out query, judge the top-20 MiniLM results.
Results are saved immediately after each query in case of timeout.

In [ ]:
judgements_path = RESULT_DIR / 'llm_judgements.json'

# ── Skip if already done ──────────────────────────────────────────────────────
if judgements_path.exists():
    print('[Judge] Loading existing judgements from disk...')
    with open(judgements_path) as f:
        all_judgements = json.load(f)
    print(f'[Judge] Loaded {len(all_judgements)} judgements')

else:
    print(f'[Judge] Running LLM judgements for {len(held_out_data)} queries...')
    print(f'[Judge] Top-20 companies per query = {len(held_out_data)*20} total LLM calls')
    print(f'[Judge] Estimated time: ~{len(held_out_data)*20*2//60} minutes')
    print('-' * 60)

    all_judgements = []
    total_start    = time.time()

    for qi, item in enumerate(held_out_data):
        qid   = item['query_id']
        query = item['query']

        # Get MiniLM top-20 for this query
        top20 = (
            minilm_df[minilm_df['query_id'] == qid]
            .sort_values('rank')
            .head(20)
        )

        print(f'\n[Judge] Query {qi+1}/20: "{query}"')
        query_judgements = []

        for _, row in top20.iterrows():
            t0             = time.perf_counter()
            score, reason  = judge_relevance(
                query,
                str(row.get('name', '')),
                str(row.get('summary', ''))
            )
            ms = (time.perf_counter() - t0) * 1000

            judgement = {
                'query_id': qid,
                'query':    query,
                'rank':     int(row['rank']),
                'domain':   row['domain'],
                'name':     str(row.get('name', '')),
                'score':    score,
                'reason':   reason,
                'ms':       round(ms, 0),
            }
            query_judgements.append(judgement)
            all_judgements.append(judgement)
            print(f'  Rank {int(row["rank"]):>4} | Score {score}/2 | {str(row.get("name",""))[:35]:<35} | {reason[:60]}')

        # Save after every query in case of timeout
        with open(judgements_path, 'w') as f:
            json.dump(all_judgements, f, indent=2)

        relevant_count = sum(1 for j in query_judgements if j['score'] >= 1)
        highly_relevant = sum(1 for j in query_judgements if j['score'] == 2)
        print(f'  → {highly_relevant} highly relevant, {relevant_count} total relevant in top-20')

    total_time = time.time() - total_start
    print(f'\n[Judge] Done! Total time: {total_time/60:.1f} minutes')
    print(f'[Judge] Saved to: {judgements_path}')

## 6 · Inspect Judgement Quality

Before computing metrics, review what the LLM judged.
Check for obvious errors — if the LLM is scoring investment banks as
highly relevant for "software companies", the judgements are not trustworthy.

In [ ]:
judgements_df = pd.DataFrame(all_judgements)

print('[Inspect] === JUDGEMENT SUMMARY ===')
print(f'  Total judgements     : {len(judgements_df)}')
print(f'  Score distribution   :')
for score in [0, 1, 2]:
    n   = (judgements_df['score'] == score).sum()
    pct = 100 * n / len(judgements_df)
    label = {0: 'Not relevant ', 1: 'Partial      ', 2: 'Highly relev.'}[score]
    print(f'    Score {score} ({label}): {n:>4} ({pct:.1f}%)')

# Compare LLM labels vs pseudo-relevance labels
print(f'\n[Inspect] === LLM vs PSEUDO-RELEVANCE COMPARISON ===')
pseudo_relevant = set()
for item in held_out_data:
    qid = item['query_id']
    prod_top100 = set(
        production_df[
            (production_df['query_id'] == qid) &
            (production_df['rank'] <= 100)
        ]['domain'].tolist()
    )
    for _, row in judgements_df[judgements_df['query_id'] == qid].iterrows():
        pseudo_relevant.add((qid, row['domain']))

agree      = 0
llm_only   = 0  # LLM says relevant, pseudo says not
pseudo_only = 0  # pseudo says relevant, LLM says not

for _, row in judgements_df.iterrows():
    qid    = row['query_id']
    domain = row['domain']
    prod_top100 = set(
        production_df[
            (production_df['query_id'] == qid) &
            (production_df['rank'] <= 100)
        ]['domain'].tolist()
    )
    llm_rel    = row['score'] >= 1
    pseudo_rel = domain in prod_top100

    if llm_rel and pseudo_rel:     agree      += 1
    elif llm_rel and not pseudo_rel: llm_only  += 1
    elif not llm_rel and pseudo_rel: pseudo_only += 1

print(f'  Both LLM and pseudo agree relevant  : {agree}')
print(f'  LLM says relevant, pseudo says not  : {llm_only}  ← companies your system found that production missed')
print(f'  Pseudo says relevant, LLM says not  : {pseudo_only}  ← production ranked but LLM disagrees')
print()
print('[Inspect] === SAMPLE JUDGEMENTS ===')
for qid in held_out_qids[:3]:
    q_judge = judgements_df[judgements_df['query_id'] == qid]
    query   = q_judge.iloc[0]['query']
    print(f'\nQuery: "{query}"')
    for _, row in q_judge.head(5).iterrows():
        print(f'  Rank {int(row["rank"]):>3} | Score {row["score"]}/2 | {str(row["name"])[:40]}')
        print(f'         Reason: {row["reason"][:80]}')

## 7 · Compute Metrics with LLM Relevance Labels

Recompute NDCG, Precision, Recall, F1 using LLM judgements as ground truth
instead of pseudo-relevance labels.

**Key differences:**
- Relevance is graded (0, 1, 2) not binary — NDCG uses graded relevance
- Only top-20 are judged — positions 21-1000 assumed not relevant for this eval
- Primary metric is NDCG@100 since Istari cares about top-1000 quality

**Two evaluations:**
1. MiniLM results with LLM labels — true quality of your best system
2. MiniLM results with pseudo labels — what you measured before

Comparing these shows how much pseudo-relevance was under/over-estimating.

In [ ]:
def ndcg_graded_at_k(retrieved_domains, judgements_dict, k):
    """
    NDCG with graded relevance (0, 1, 2).
    judgements_dict: {domain: score} for judged companies.
    Unjudged companies assumed score=0.
    """
    def dcg(domains, k):
        return sum(
            judgements_dict.get(d, 0) / np.log2(i + 2)
            for i, d in enumerate(domains[:k])
        )
    ideal_domains = sorted(judgements_dict.keys(),
                           key=lambda d: -judgements_dict[d])
    ideal = dcg(ideal_domains, k)
    return dcg(retrieved_domains, k) / ideal if ideal > 0 else 0

def precision_llm_at_k(retrieved, judgements_dict, k, threshold=1):
    """Precision using LLM labels. threshold=1 means score>=1 is relevant."""
    top_k    = retrieved[:k]
    relevant = sum(1 for d in top_k if judgements_dict.get(d, 0) >= threshold)
    return relevant / k if k else 0

def recall_llm_at_k(retrieved, judgements_dict, k, threshold=1):
    total_relevant = sum(1 for s in judgements_dict.values() if s >= threshold)
    if total_relevant == 0:
        return 0
    top_k    = retrieved[:k]
    relevant = sum(1 for d in top_k if judgements_dict.get(d, 0) >= threshold)
    return relevant / total_relevant

# ── Build judgement dicts per query ──────────────────────────────────────────
query_judgement_dicts = {}
for item in held_out_data:
    qid = item['query_id']
    q_judge = judgements_df[judgements_df['query_id'] == qid]
    query_judgement_dicts[qid] = dict(zip(q_judge['domain'], q_judge['score']))

# ── Evaluate MiniLM with LLM labels ──────────────────────────────────────────
print('[Eval] Computing metrics with LLM relevance labels...')
llm_eval_rows = []

for item in held_out_data:
    qid          = item['query_id']
    query        = item['query']
    judgements   = query_judgement_dicts[qid]
    retrieved    = (
        minilm_df[minilm_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )

    for k in K_VALUES:
        llm_eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'ndcg':      ndcg_graded_at_k(retrieved, judgements, k),
            'precision': precision_llm_at_k(retrieved, judgements, k),
            'recall':    recall_llm_at_k(retrieved, judgements, k),
        })

llm_eval_df = pd.DataFrame(llm_eval_rows)
llm_eval_df.to_csv(RESULT_DIR / 'evaluation_llm_labels.csv', index=False)

# ── Print results ─────────────────────────────────────────────────────────────
print('\n[Eval] === MiniLM — LLM RELEVANCE LABELS (True Quality) ===')
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8}')
print('  ' + '-' * 36)
for k in K_VALUES:
    sub = llm_eval_df[llm_eval_df['k'] == k]
    print(f'  {k:<6} {sub["ndcg"].mean():>8.3f} {sub["precision"].mean():>10.3f} {sub["recall"].mean():>8.3f}')

## 8 · Comparison: Pseudo-Relevance vs LLM Labels

The key insight — how much was pseudo-relevance under/over-estimating your system?

In [ ]:
# Pseudo-relevance evaluation on same 20 held-out queries
print('[Compare] Computing pseudo-relevance metrics on held-out queries...')

def get_pseudo_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def ndcg_binary_at_k(retrieved, relevant, k):
    ideal = sum(1/np.log2(i+2) for i in range(min(len(relevant), k)))
    actual = sum(1/np.log2(i+2) for i,d in enumerate(retrieved[:k]) if d in relevant)
    return actual/ideal if ideal > 0 else 0

pseudo_eval_rows = []
for item in held_out_data:
    qid       = item['query_id']
    relevant  = get_pseudo_relevant(qid)
    retrieved = (
        minilm_df[minilm_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        n_rel = len(set(retrieved[:k]) & relevant)
        pseudo_eval_rows.append({
            'query_id':  qid, 'k': k,
            'ndcg':      ndcg_binary_at_k(retrieved, relevant, k),
            'precision': n_rel / k if k else 0,
            'recall':    n_rel / len(relevant) if relevant else 0,
        })

pseudo_eval_df = pd.DataFrame(pseudo_eval_rows)

print('\n[Compare] =======================================================')
print('[Compare] PSEUDO-RELEVANCE vs LLM LABELS — MiniLM all fields')
print('[Compare] =======================================================')
print(f'  {"k":<6} {"Pseudo NDCG":>12} {"LLM NDCG":>10} {"Difference":>12}')
print('  ' + '-' * 44)
for k in K_VALUES:
    p_ndcg = pseudo_eval_df[pseudo_eval_df['k']==k]['ndcg'].mean()
    l_ndcg = llm_eval_df[llm_eval_df['k']==k]['ndcg'].mean()
    diff   = l_ndcg - p_ndcg
    arrow  = '↑' if diff > 0 else '↓'
    print(f'  {k:<6} {p_ndcg:>12.3f} {l_ndcg:>10.3f} {arrow}{abs(diff):>10.3f}')

print(f'\n[Compare] Primary metric NDCG@{PRIMARY_K}:')
p100 = pseudo_eval_df[pseudo_eval_df['k']==PRIMARY_K]['ndcg'].mean()
l100 = llm_eval_df[llm_eval_df['k']==PRIMARY_K]['ndcg'].mean()
print(f'  Pseudo-relevance : {p100:.3f}')
print(f'  LLM labels       : {l100:.3f}')
if l100 > p100:
    print(f'  → LLM labels show {100*(l100-p100)/p100:.1f}% HIGHER quality')
    print(f'  → Pseudo-relevance was UNDERESTIMATING your system')
else:
    print(f'  → LLM labels show {100*(p100-l100)/p100:.1f}% LOWER quality')
    print(f'  → Pseudo-relevance was OVERESTIMATING your system')

# Save comparison
comp_rows = []
for k in K_VALUES:
    p = pseudo_eval_df[pseudo_eval_df['k']==k]['ndcg'].mean()
    l = llm_eval_df[llm_eval_df['k']==k]['ndcg'].mean()
    comp_rows.append({'k':k,'pseudo_ndcg':round(p,3),'llm_ndcg':round(l,3),'diff':round(l-p,3)})
pd.DataFrame(comp_rows).to_csv(RESULT_DIR/'comparison_pseudo_vs_llm.csv',index=False)
print(f'\n[Compare] Saved to result/08_llm_relevance_judge/comparison_pseudo_vs_llm.csv')

## 9 · Key Findings & Next Steps

In [ ]:
print('[Findings] ============================================================')
print('[Findings] KEY FINDINGS — LLM RELEVANCE EVALUATION')
print('[Findings] ============================================================')

llm_n10  = llm_eval_df[llm_eval_df['k']==10]['ndcg'].mean()
llm_n100 = llm_eval_df[llm_eval_df['k']==100]['ndcg'].mean()
ps_n100  = pseudo_eval_df[pseudo_eval_df['k']==100]['ndcg'].mean()

print(f'\n  MiniLM NDCG@10  (LLM labels)    : {llm_n10:.3f}')
print(f'  MiniLM NDCG@100 (LLM labels)    : {llm_n100:.3f}')
print(f'  MiniLM NDCG@100 (pseudo labels) : {ps_n100:.3f}')

score_dist = judgements_df['score'].value_counts().sort_index()
total      = len(judgements_df)
print(f'\n  Relevance distribution in top-20:')
for score, count in score_dist.items():
    label = {0:'Not relevant', 1:'Partial', 2:'Highly relevant'}[score]
    print(f'    {label:<18}: {count:>4} ({100*count/total:.1f}%)')

print(f'\n  LLM vs pseudo agreement analysis saved to llm_judgements.json')
print(f'  → Use held_out_queries.json and train_queries.json for Option B (fine-tuning)')
print('[Findings] ============================================================')